# Subsetting Data with the GDEX API

---

## Overview

GDEX supports server-side subsetting by time range, spatial region, and variable. This notebook shows how to build and submit a subset request using the API.

1. Retrieve subsettable parameters via the control file template
2. Build a JSON control file
3. Submit a subset request
4. Check status and retrieve output files

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| api_intro | Necessary | GDEX API conventions |
| api_metadata | Helpful | Know your dataset's variables and coverage |
| GDEX account and token | Necessary | Register at https://gdex.ucar.edu |

- **Time to learn**: 25 minutes

---

## Imports

In [1]:
import requests
import json
import os
import time

## Retrieving Subsettable Parameters

`GET /control_file_template/{dsid}/` returns the JSON control file structure. `GET /paramsummary/{dsid}/` summarises available parameters and date ranges.


In [2]:
BASE_URL = "https://api.gdex.ucar.edu"
DSID = "d084001"
TOKEN = os.environ.get("GDEX_TOKEN", "your_token_here")

template = requests.get(f"{BASE_URL}/control_file_template/{DSID}/").json()
print(json.dumps(template, indent=2))


{
  "status": "ok",
  "http_response": 200,
  "error_messages": [],
  "data": {
    "template": "dataset=ds084.1\ndate=201609200000/to/201609200000\ndatetype=init\nparam=TMP/R H/ABS V\nlevel=ISBL:850/700/500\n#oformat=netCDF\nnlat=30\nslat=-25\nwlon=-150\nelon=-30\nproduct=Analysis/12-hour Forecast/6-hour Forecast/18-hour Forecast\ntargetdir=/glade/scratch\n"
  },
  "contact": "datahelp@ucar.edu"
}


:::{tip}
Store your GDEX token in an environment variable rather than hardcoding it:

```bash
export GDEX_TOKEN="your_token_here"
```

Retrieve your token from your [GDEX profile page](https://gdex.ucar.edu/accounts/profile/).
:::


## Building the Control File

Populate the control file fields: dataset ID, date range, spatial bounds, and selected variables/parameters.

In [4]:
control_file = {
    "dsid": DSID,
    "startdate": "2020-01-01 00:00",
    "enddate": "2020-01-31 23:59",
    # ... parameters from template
}

your_token_here


## Submitting and Monitoring

`POST /submit_json/?token={token}` submits the request and returns an `rindex`. Poll `GET /status/{rindex}/?token={token}` until the status is `Completed`.


In [6]:
if TOKEN != 'your_token_here':
    resp   = requests.post(f"{BASE_URL}/submit_json/?token={TOKEN}", json=control_file)
    rindex = resp.json()["data"]["rindex"]
    print(f"Request submitted: rindex = {rindex}")

    while True:
        status = requests.get(f"{BASE_URL}/status/{rindex}/?token={TOKEN}").json()
        print(status["data"]["status"])
        if status["data"]["status"] in ("Completed", "Error"):
            break
        time.sleep(60)
else:
    print('Remember to set your TOKEN!')


Remember to set your TOKEN!


:::{tip}
Once you have downloaded your output files, call `DELETE /purge/{rindex}/?token={TOKEN}` to remove the request from the server and free up your storage quota.
:::


## Retrieving Output Files

`GET /get_req_files/{rindex}/?token={token}` lists the resulting files. Download them or transfer via Globus.


In [7]:
if TOKEN != 'your_token_here':
    files = requests.get(f"{BASE_URL}/get_req_files/{rindex}/?token={TOKEN}").json()
    for f in files["data"]:
        print(f)
else:
    print('Remember to set your TOKEN!')

Remember to set your TOKEN!


---

## Summary

GDEX subset requests are asynchronous: submit a JSON control file, poll for completion, then retrieve the resulting files. Use `DELETE /api/purge/{rindex}/` to clean up old requests.


:::{seealso}
- {doc}`metadata` — query dataset metadata before building a subset request
- {doc}`globus_intro` — transfer large results via Globus instead of HTTP
:::


## Resources and References

- [GDEX API Documentation](https://api.gdex.ucar.edu/documentation/)
- [GDEX Profile & Token](https://gdex.ucar.edu/accounts/profile/)
